In [ ]:
"""
Script to analyze Aegis-D inference results
Used to count the distribution of assessment and violated_categories
"""

import json
import os
import argparse
from collections import Counter, defaultdict
from pathlib import Path

# Load exclusion lists
exclude_ids_by_task = {
    "vague": ["3I02618YA14S7SHVOPAYBZTIDARUPP", "3KIBXJ1WD6SWJW0IFBTHGCFU1SKOKK", "3NPI0JQDAP3D7F26OKKO637GUJPPT8", "3SITXWYCNW7IK2AGAP3K0MNXQ3DBXR"],
    "reverse": ["39L1G8WVWRP5R6LAO337NULKX9931E"],
    "biased": [],
    "fake": ["9929"]
}

def load_results_from_directory(results_dir, task, resp_flag):
    """Load all JSON result files from the directory"""
    results = []
    json_files = list(Path(results_dir).glob("result_*.json"))
    
    print(f"Found {len(json_files)} result files")
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                results.append(data)
        except Exception as e:
            print(f"Error loading file {json_file}: {e}")
    
    # Exclude results based on task-specific criteria
    if resp_flag:
        if task in exclude_ids_by_task:
            exclude_ids = set(exclude_ids_by_task[task])
            results = [r for r in results if r.get('sample_id') not in exclude_ids]
    
    return results


def analyze_assessments(results):
    """Analyze assessment distribution"""
    assessments = []
    for result in results:
        parsed_response = result.get('parsed_response', {})
        parsed_response = parsed_response.get('parsed_response', {})
        assessment = parsed_response.get('safety_label', 'unknown')
        assessments.append(assessment)
    
    assessment_counts = Counter(assessments)
    return assessment_counts


def analyze_violated_categories(results):
    """Analyze violated_categories distribution"""
    all_categories = []
    category_combinations = []
    
    for result in results:
        parsed_response = result.get('parsed_response', {})
        parsed_response = parsed_response.get('parsed_response', {})
        category = parsed_response.get('category', [])
        
        # # Record single categories
        # for category in violated_categories:
        #     all_categories.append(category)
        all_categories.append(category)
        
        # Record category combinations
        if category:
            combo = sorted(category)
            category_combinations.append(tuple(combo))
        else:
            category_combinations.append(('None',))
    
    single_category_counts = Counter(all_categories)
    combination_counts = Counter(category_combinations)
    
    return single_category_counts, combination_counts


def print_statistics(results):
    """Print statistics"""
    print(f"\n{'='*60}")
    print(f"Overall Statistics - Total {len(results)} samples")
    print(f"{'='*60}")
    
    # 1. Assessment distribution
    assessment_counts = analyze_assessments(results)
    print("Assessment Distribution:")
    print("-" * 30)
    total_samples = sum(assessment_counts.values())
    for assessment, count in assessment_counts.most_common():
        percentage = (count / total_samples) * 100
        print(f"  {assessment:10}: {count:4} ({percentage:5.1f}%)")
    
    # # 2. Violated Categories analysis
    # single_counts, combo_counts = analyze_violated_categories(results)
    
    # print("Single Category Statistics:")
    # print("-" * 30)
    # if single_counts:
    #     for category, count in single_counts.most_common():
    #         print(f"  {category}: {count}")
    # else:
    #     print("  No violated categories")
    
    # print("Category Combination Statistics (Top 10):")
    # print("-" * 40)
    # for combo, count in combo_counts.most_common(10):
    #     combo_str = ", ".join(combo) if combo != ('None',) else "None"
    #     print(f"  {combo_str}: {count}")



In [ ]:
for task in ["vague", "reverse", "fake", "biased"]:
    for model in ["lguard_ori_resp", "lguard_ori", "lguard_cus_resp", "lguard_cus"]:
        if model.endswith("resp"):
            resp_flag = True
        else:
            resp_flag = False
        results_dir = f'./results_{task}_{model}'

        if not os.path.exists(results_dir):
            print(f"Error: Directory {results_dir} does not exist")

        # Load results
        print(f"Loading results from {results_dir} ...")
        results = load_results_from_directory(results_dir, task, resp_flag)

        if not results:
            print("Error: No valid result files found")

        # Print statistics
        print_statistics(results)

Loading results from ./results_vague_lguard_ori_resp ...
Found 2500 result files

Overall Statistics - Total 2496 samples
Assessment Distribution:
------------------------------
  safe      : 2033 ( 81.5%)
  unsafe    :  463 ( 18.5%)
Loading results from ./results_reverse_lguard_ori_resp ...
Found 2500 result files

Overall Statistics - Total 2499 samples
Assessment Distribution:
------------------------------
  safe      : 2473 ( 99.0%)
  unsafe    :   26 (  1.0%)
Loading results from ./results_fake_lguard_ori_resp ...
Found 2800 result files

Overall Statistics - Total 2799 samples
Assessment Distribution:
------------------------------
  safe      : 2645 ( 94.5%)
  unsafe    :  154 (  5.5%)
Loading results from ./results_biased_lguard_ori_resp ...
Found 2800 result files

Overall Statistics - Total 2800 samples
Assessment Distribution:
------------------------------
  safe      : 2154 ( 76.9%)
  unsafe    :  646 ( 23.1%)
